## Code for sampling structures and processing predicted models for AF-MM benchmarking of new interface types

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
import scipy.stats
import os, requests, random, json
pd.options.mode.copy_on_write = True

## Read in data describing PDB interface clusters and the first and last deposition dates for each cluster

In [2]:
pdb_ids_cluster_first_last = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/pdb_ids_cluster_first_last.tsv", header=None, sep="\t")
pdb_ids_cluster_first_last.columns = ["cluster_id","first_date","last_date","pdb_id_list"]
pdb_ids_cluster_first_last

,cluster_id,first_date,last_date,pdb_id_list
0,0,20130227.0,20221003.0,"3zph,4c9s,4c9t,4d06,4d4f,8b7r,8b7u,8b7z"
1,1,20120327.0,20240510.0,"4edw,4edx,7k8s,7k8t,8euq,8vsj,8zhd,9b7b"
2,2,20240327.0,20240327.0,9b7d
3,3,20240327.0,20240327.0,9b7d
4,4,20240327.0,20240327.0,9b7d
...,...,...,...,...
77162,77162,20201211.0,20210618.0,"7b7u,7f4g"
77163,77163,20190625.0,20220928.0,"6s3k,8b70,8b71"
77164,77164,20220929.0,20220929.0,8b7f
77165,77165,20220930.0,20220930.0,8b7h


In [3]:
print("Number of clusters beyond the training date of AF-MM v2: ", pdb_ids_cluster_first_last[pdb_ids_cluster_first_last["first_date"] > 20180430].shape[0])
print("Number of clusters beyond the training date of AF3: ", pdb_ids_cluster_first_last[pdb_ids_cluster_first_last["first_date"] > 20210930].shape[0])
af2_new_cluster_ids = list(set(pdb_ids_cluster_first_last[pdb_ids_cluster_first_last["first_date"] > 20180430].cluster_id))
af3_new_cluster_ids = list(set(pdb_ids_cluster_first_last[pdb_ids_cluster_first_last["first_date"] > 20210930].cluster_id))

Number of clusters beyond the training date of AF-MM v2:  38031
Number of clusters beyond the training date of AF3:  20103


In [4]:
pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")
pdb_clusters["chain_id_0"] = pdb_clusters["chain_id_0"].fillna('NA')
pdb_clusters["chain_id_1"] = pdb_clusters["chain_id_1"].fillna('NA')

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_10890/4260261053.py:1: DtypeWarning: Columns (32,34) have mixed types. Specify dtype option on import or set low_memory=False.
  pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")


In [5]:
num_if_per_pdb = pdb_clusters.groupby("pdb_id")["old_complex_id"].size()
single_dimers = list(set(num_if_per_pdb[num_if_per_pdb == 1].index))
print(len(single_dimers))
pdb_clusters.loc[:, "single_dimer"] = 0
pdb_clusters.loc[pdb_clusters["pdb_id"].isin(single_dimers), "single_dimer"] = 1
print(pdb_clusters[pdb_clusters["single_dimer"] == 1].shape[0])

97200
97200


## Filter and random selection

In [ ]:
# Include only interface cluster representatives
pdb_clusters_filtered_1=pdb_clusters[pdb_clusters["intrep"]==1]
print(len(pdb_clusters_filtered_1))

# Restrict to post-training clsuters
filter_intcluster_isin_af3=pdb_clusters_filtered_1["intcluster"].isin(af3_new_cluster_ids)
filter_intcluster_isin_af3_1=pdb_clusters["intcluster"].isin(af3_new_cluster_ids)
pdb_clusters_isin_af3=pdb_clusters[filter_intcluster_isin_af3_1]
pdb_clusters_filtered_2=pdb_clusters_filtered_1[filter_intcluster_isin_af3]
print(len(pdb_clusters_filtered_2))

# Include only dimers with combined length < 1000 residues to reduce computational cost
pdb_clusters_filtered_3=pdb_clusters_filtered_2[pdb_clusters_filtered_2["dimer_len"]<1000]
print(len(pdb_clusters_filtered_3))

# Exclude peptide-peptide interfaces
mask_4=(pdb_clusters_filtered_3["protpep_status"]== "Peptide-peptide")
pdb_clusters_filtered_4=pdb_clusters_filtered_3[~mask_4]
print(len(pdb_clusters_filtered_4))

# Exclude protein-protein interfaces which are in the bottom quartile of interface size
mask_5=(pdb_clusters_filtered_4["protpep_status"]=="Protein-protein")&(pdb_clusters_filtered_4["if_size_cat"]== "X-Small")
pdb_clusters_filtered_5=pdb_clusters_filtered_4[~mask_5]
print(len(pdb_clusters_filtered_5))

# Repeat filtering for pre-training clusters
mask=filter_intcluster_isin_af3_1 
pdb_clusters_not_in_af3=pdb_clusters[~filter_intcluster_isin_af3_1]

pdb_clusters_PRE_filtered_2=pdb_clusters_not_in_af3[pdb_clusters_not_in_af3["intrep"]==1]
print(len(pdb_clusters_PRE_filtered_2))

pdb_clusters_PRE_filtered_3=pdb_clusters_PRE_filtered_2[pdb_clusters_PRE_filtered_2["dimer_len"]<1000]
print(len(pdb_clusters_PRE_filtered_3))

mask_4_PRE=(pdb_clusters_PRE_filtered_3["protpep_status"]== "Peptide-peptide")
pdb_clusters_PRE_filtered_4=pdb_clusters_PRE_filtered_3[~mask_4_PRE]
len(pdb_clusters_PRE_filtered_4)

mask_5_PRE=(pdb_clusters_PRE_filtered_4["protpep_status"]=="Protein-protein")&(pdb_clusters_PRE_filtered_4["if_size_cat"]== "X-Small")
pdb_clusters_PRE_filtered_5=pdb_clusters_PRE_filtered_4[~mask_5_PRE]
len(pdb_clusters_PRE_filtered_5)

# Random selection, with even selection from different orderedness categories
random_sele_filter5_order_order=pdb_clusters_filtered_5[pdb_clusters_filtered_5["if_type"]=="Order-order"].sample(n=300,random_state=1)
random_sele_filter5_order_order.loc[:,"AFstatus"] = "Post-training"
random_sele_filter5_disorder_order=pdb_clusters_filtered_5[pdb_clusters_filtered_5["if_type"]=="Disorder-order"].sample(n=300,random_state=2)
random_sele_filter5_disorder_order.loc[:,"AFstatus"] = "Post-training"
random_sele_filter5_disorder_disorder=pdb_clusters_filtered_5[pdb_clusters_filtered_5["if_type"]=="Disorder-disorder"].sample(n=300, random_state=3)
random_sele_filter5_disorder_disorder.loc[:,"AFstatus"] = "Post-training"
random_sele_filter5_PRE_order_order=pdb_clusters_PRE_filtered_5[pdb_clusters_PRE_filtered_5["if_type"]=="Order-order"].sample(n=300,random_state=4)
random_sele_filter5_PRE_order_order.loc[:,"AFstatus"] = "Pre-training"
random_sele_filter5_PRE_disorder_order=pdb_clusters_PRE_filtered_5[pdb_clusters_PRE_filtered_5["if_type"]=="Disorder-order"].sample(n=300,random_state=5)
random_sele_filter5_PRE_disorder_order.loc[:,"AFstatus"] = "Pre-training"
random_sele_filter5_PRE_disorder_disorder=pdb_clusters_PRE_filtered_5[pdb_clusters_PRE_filtered_5["if_type"]=="Disorder-disorder"].sample(n=300,random_state=6)
random_sele_filter5_PRE_disorder_disorder.loc[:,"AFstatus"] = "Pre-training"

selected_ifs = pd.concat([random_sele_filter5_order_order,random_sele_filter5_disorder_order,random_sele_filter5_disorder_disorder,random_sele_filter5_PRE_order_order,random_sele_filter5_PRE_disorder_order,random_sele_filter5_PRE_disorder_disorder], axis=0)
print(len(selected_ifs))

77167
20103
16695
16615
12346
57064
50944
1800


In [7]:
selected_ifs["report_lookup_id"] = selected_ifs[['old_complex_id', 'pdb_id']].apply(lambda x: "DI"+str(x.iloc[0])+"_"+str(x.iloc[1]), axis=1)
selected_ifs.loc[:,"dataset"] = "Original"
selected_ifs

,old_complex_id,chain_id_0,pdb_id,chain_id_1,new_complex_id,dimercluster,intcluster,dimerrep,intrep,dimer_len,...,pdb_expt_method,pdb_resolution,pdb_title,gene_names_0,gene_names_1,go_cat,single_dimer,AFstatus,report_lookup_id,dataset
2840068,208129618,A,8tid-assembly1,F,2287509,167130,19901,1,1,923,...,ELECTRON MICROSCOPY,3.60,Combined linker domain of N-DRC and associated...,TTHERM_01345750,TTHERM_00697450,"GO:0003352,GO:0031514",0,Post-training,DI208129618_8tid-assembly1,Original
2816467,198367644,A-43,8s5d-assembly1,A-117,2154452,104527,18693,1,1,330,...,ELECTRON MICROSCOPY,3.79,Cryo-EM structure of Arf1-decorated membrane t...,"ARF1,D1244,YDL192W","ARF1,D1244,YDL192W",GO:0016192,0,Post-training,DI198367644_8s5d-assembly1,Original
2623675,62511604,B9,8gzu-assembly1,N5,898016,19214,72069,1,1,901,...,ELECTRON MICROSCOPY,4.18,Cryo-EM structure of Tetrahymena thermophila r...,TTHERM_00985010,nad5,GO:0015990,0,Post-training,DI62511604_8gzu-assembly1,Original
2571703,45569345,B,8faz-assembly1,D,706743,84585,34644,1,1,380,...,ELECTRON MICROSCOPY,2.30,Cryo-EM structure of the human BCDX2 complex,"RAD51B,RAD51L1,REC2","RAD51D,RAD51L3",NaN,0,Post-training,DI45569345_8faz-assembly1,Original
2587211,57535053,0T,8g2z-assembly1,MB,806252,59693,5921,1,1,695,...,ELECTRON MICROSCOPY,4.10,48-nm doublet microtubule from Tetrahymena the...,TTHERM_00348390,"BTU1,BTU2",GO:0007017,0,Post-training,DI57535053_8g2z-assembly1,Original
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1420639,100576450,0,6lqm-assembly1,Y,1431022,188702,44492,1,1,223,...,ELECTRON MICROSCOPY,3.09,Cryo-EM structure of a pre-60S ribosomal subun...,"ZNF622,ZPR9",RPL17,"GO:0042273,GO:0006412",0,Pre-training,DI100576450_6lqm-assembly1,Original
2019648,89942314,W,7kor-assembly1,Y,1325422,162067,50449,1,1,307,...,ELECTRON MICROSCOPY,7.80,Structure of cardiac native thin filament at p...,TPM1,TPM1,GO:0060048,0,Pre-training,DI89942314_7kor-assembly1,Original
604791,151759571,A,3nyl-assembly1,A-2,1675930,22454,42817,1,1,396,...,X-RAY DIFFRACTION,2.80,The X-ray structure of an antiparallel dimer o...,"A4,AD1,APP","A4,AD1,APP",NaN,1,Pre-training,DI151759571_3nyl-assembly1,Original
2576627,56120747,SN,8fkq-assembly1,ST,755488,111672,2527,1,1,208,...,ELECTRON MICROSCOPY,2.76,Human nucleolar pre-60S ribosomal subunit (Sta...,"EBNA1BP2,EBP2","KIAA0112,RRR,RRS1","GO:0006364,GO:0042273",0,Pre-training,DI56120747_8fkq-assembly1,Original


In [ ]:
# Resampling to increase number of interfaces from assemblies with only one interface (dimer assemblies)
mask_6=pdb_clusters_filtered_5["single_dimer"] == 1
pdb_clusters_filtered_6 = pdb_clusters_filtered_5[mask_6]
print(len(pdb_clusters_filtered_6))

mask_7 = pdb_clusters_filtered_6["old_complex_id"].isin(list(set(selected_ifs.old_complex_id)))
pdb_clusters_filtered_7 = pdb_clusters_filtered_6[~mask_7]
print(len(pdb_clusters_filtered_7))

print(pdb_clusters_filtered_7["if_type"].value_counts())

mask_6_PRE = pdb_clusters_PRE_filtered_5["single_dimer"] == 1
pdb_clusters_PRE_filtered_6 = pdb_clusters_PRE_filtered_5[mask_6_PRE]
print(len(pdb_clusters_PRE_filtered_6))

mask_7_PRE = pdb_clusters_PRE_filtered_6["old_complex_id"].isin(list(set(selected_ifs.old_complex_id)))
pdb_clusters_PRE_filtered_7 = pdb_clusters_PRE_filtered_6[~mask_7_PRE]
print(len(pdb_clusters_PRE_filtered_7))

print(pdb_clusters_PRE_filtered_7["if_type"].value_counts())

random_sele_filter7_order_order=pdb_clusters_filtered_7[pdb_clusters_filtered_7["if_type"]=="Order-order"].sample(n=61,random_state=1)
random_sele_filter7_order_order.loc[:,"AFstatus"] = "Post-training"
random_sele_filter7_disorder_order=pdb_clusters_filtered_7[pdb_clusters_filtered_7["if_type"]=="Disorder-order"].sample(n=72,random_state=2)
random_sele_filter7_disorder_order.loc[:,"AFstatus"] = "Post-training"
random_sele_filter7_disorder_disorder=pdb_clusters_filtered_7[pdb_clusters_filtered_7["if_type"]=="Disorder-disorder"].sample(n=2, random_state=3)
random_sele_filter7_disorder_disorder.loc[:,"AFstatus"] = "Post-training"
random_sele_filter7_PRE_order_order=pdb_clusters_PRE_filtered_7[pdb_clusters_PRE_filtered_7["if_type"]=="Order-order"].sample(n=15,random_state=4)
random_sele_filter7_PRE_order_order.loc[:,"AFstatus"] = "Pre-training"
random_sele_filter7_PRE_disorder_order=pdb_clusters_PRE_filtered_7[pdb_clusters_PRE_filtered_7["if_type"]=="Disorder-order"].sample(n=64,random_state=5)
random_sele_filter7_PRE_disorder_order.loc[:,"AFstatus"] = "Pre-training"

selected_ifs_2 = pd.concat([random_sele_filter7_order_order,random_sele_filter7_disorder_order,random_sele_filter7_disorder_disorder,random_sele_filter7_PRE_order_order,random_sele_filter7_PRE_disorder_order], axis=0)
selected_ifs_2["report_lookup_id"] = selected_ifs_2[['old_complex_id', 'pdb_id']].apply(lambda x: "DI"+str(x.iloc[0])+"_"+str(x.iloc[1]), axis=1)
selected_ifs_2.loc[:,"dataset"] = "Second"
print(len(selected_ifs_2))

1232
1158
if_type
Order-order          744
Unannotated          340
Disorder-order        72
Disorder-disorder      2
Name: count, dtype: int64
10849
10622
if_type
Order-order          8630
Unannotated          1191
Disorder-order        636
Disorder-disorder     165
Name: count, dtype: int64
214


In [9]:
selected_ifs_concat = pd.concat([selected_ifs, selected_ifs_2], axis=0)
selected_ifs_concat.reset_index(inplace=True, drop=True)

## Create AF input files

In [12]:
def make_fasta_file(filepath, name, chains, sequences):
    """

        Write a dictionary with chain IDs and sequences to a FASTA file.

        Input:
            filepath|str: Path to desired output directory
            name|str: Identifying name of prediction, also to be used as output filename
            chains|dict: Dictionary containing information about the input sequences, where keys are the chains IDs and values are the chains sequences

    """
    if (len(chains) != 2) or (len(sequences) != 2):
        print("Incorrect number of chains/sequences for", name, chains)
    else:
        with open(os.path.join(filepath, name+".fasta"), "a+") as f:
            for i in range(0,2):
                f.write(">" + chains[i] + "\n" + sequences[i] + "\n")

In [ ]:
# Fetch primary sequence from PDBe API, including unresolved residues
dimer_lens = []
url = "https://www.ebi.ac.uk/pdbe/api/pdb/entry/molecules/"
for i,r in selected_ifs_concat.iterrows():
    try:
        sequences = []
        dimer_len = 0
        pdb_id = r["pdb_id"][0:4]
        chain_id_0 = r["chain_id_0"].split("-")[0]
        chain_id_1 = r["chain_id_1"].split("-")[0]
        chains = sorted([chain_id_0, chain_id_1])
        response = requests.get(url+pdb_id).json()
        for chain_id in chains:
            for item in response[pdb_id]:
                if chain_id in item["in_chains"]:
                    if item["molecule_type"] == "polypeptide(L)":
                        sequences.append(item["sequence"])
                        dimer_len += len(item["sequence"])
        if r["dataset"] == "Original":
            make_fasta_file(f"/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/input_files/", r["report_lookup_id"], chains, sequences)
        elif r["dataset"] == "Second":
            make_fasta_file(f"/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/input_files/second_batch/", r["report_lookup_id"], chains, sequences)
        dimer_lens.append(dimer_len)

    except:
        print("Unable to write input file for", pdb_id, chain_id_0, chain_id_1)
        dimer_lens.append(dimer_len)

selected_ifs_concat.loc[:,"model_len"] = dimer_lens

Unable to write input file for 8t0q A B
Unable to write input file for 8pni A B
Incorrect number of chains/sequences for DI84757856_5j8p-assembly1 ['A', 'B']


In [16]:
## Fixing errors
# 8t0q A,B: unable to retrieve structure info, superceded by 9q1a
sequences = []
dimer_len = 0
pdb_id = '9q1a'
chain_id_0 = "A"
chain_id_1 = "B"
chains = sorted([chain_id_0, chain_id_1])
response = requests.get(url+pdb_id).json()
for chain_id in chains:
    for item in response[pdb_id]:
        if chain_id in item["in_chains"]:
            if item["molecule_type"] == "polypeptide(L)":
                sequences.append(item["sequence"])
                dimer_len += len(item["sequence"])
make_fasta_file(f"/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/input_files/", "DI203378256_8t0q-assembly1", chains, sequences)
selected_ifs_concat.loc[selected_ifs_concat["report_lookup_id"] == "DI203378256_8t0q-assembly1", "model_len"] = dimer_len

# 8pni A,B: unable to retrieve structure info, superceded by 9rvg
sequences = []
dimer_len = 0
pdb_id = '9rvg'
chain_id_0 = "A"
chain_id_1 = "B"
chains = sorted([chain_id_0, chain_id_1])
response = requests.get(url+pdb_id).json()
for chain_id in chains:
    for item in response[pdb_id]:
        if chain_id in item["in_chains"]:
            if item["molecule_type"] == "polypeptide(L)":
                sequences.append(item["sequence"])
                dimer_len += len(item["sequence"])
make_fasta_file(f"/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/input_files/", "DI163328866_8pni-assembly1", chains, sequences)
selected_ifs_concat.loc[selected_ifs_concat["report_lookup_id"] == "DI163328866_8pni-assembly1", "model_len"] = dimer_len

# 5j8p A,B: unable to retrieve both chains, chain B is a D-polypeptide, not L-polypeptide so was filtered out
sequences = []
dimer_len = 0
pdb_id = '5j8p'
chain_id_0 = "A"
chain_id_1 = "B"
chains = sorted([chain_id_0, chain_id_1])
response = requests.get(url+pdb_id).json()
for chain_id in chains:
    for item in response[pdb_id]:
        if chain_id in item["in_chains"]:
            if "polypeptide" in item["molecule_type"]:
                sequences.append(item["sequence"])
                dimer_len += len(item["sequence"])
make_fasta_file(f"/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/input_files/", "DI84757856_5j8p-assembly1", chains, sequences)
selected_ifs_concat.loc[selected_ifs_concat["report_lookup_id"] == "DI84757856_5j8p-assembly1", "model_len"] = dimer_len

In [17]:
selected_ifs_concat["dimer_len"] = selected_ifs_concat["old_complex_id"].map(dict(zip(pdb_clusters.old_complex_id, pdb_clusters.dimer_len)))

## Evaluate predictions

In [10]:
results_df = selected_ifs_concat[["old_complex_id","pdb_id","chain_id_0","chain_id_1","if_size_cat","AFstatus","dimer_len","model_len","report_lookup_id"]]

results_df.loc[:,"1"] = np.nan
results_df.loc[:,"2"] = np.nan
results_df.loc[:,"3"] = np.nan
results_df.loc[:,"4"] = np.nan
results_df.loc[:,"5"] = np.nan

results_df = results_df.melt(id_vars=["old_complex_id","pdb_id","chain_id_0","chain_id_1","if_size_cat","AFstatus","dimer_len","model_len","report_lookup_id"], value_vars=["1","2","3","4","5"])

results_df["model_to_nat_len_ratio"] = results_df["model_len"] / results_df["dimer_len"]

results_df.loc[:,"DockQ"] = np.nan
results_df.loc[:,"iRMSD"] = np.nan
results_df.loc[:,"lRMSD"] = np.nan
results_df.loc[:,"FNonNat"] = np.nan
results_df.loc[:,"nat_correct"] = np.nan
results_df.loc[:,"nat_total"] = np.nan
results_df.loc[:,"nonnat_count"] = np.nan
results_df.loc[:,"model_total"] = np.nan
results_df.loc[:,"iptm"] = np.nan

results_df.drop("value", axis=1, inplace=True)
results_df.rename(columns={"variable":"rank"}, inplace=True)

results_df

KeyError: "['model_len'] not in index"

In [19]:
results_df["model_to_nat_len_ratio"].describe()

count    10070.000000
mean         1.460423
std          1.495600
min          0.900000
25%          1.016548
50%          1.093724
75%          1.296296
max         30.235772
Name: model_to_nat_len_ratio, dtype: float64

In [20]:
import os

filelist = os.listdir("/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/output_files")
filelist = [x for x in filelist if ((x[0:2] == "DI") & (len(x.split("_")) > 2))]
modellist = [x for x in filelist if x.split("_")[2] == "unrelaxed"]
ranklist = ["_".join(x.split("_")[0:5]) for x in modellist]
modelnumlist = [x.split("_")[9] for x in modellist]
rank_to_model = dict(zip(ranklist, modelnumlist))
print(rank_to_model)
#predlist = list(set(["_".join(x.split("_")[0:2]) for x in filelist]))
#print(len(predlist))

{'DI99777053_7lgh-assembly1_unrelaxed_rank_005': '5', 'DI100047846_4lij-assembly2_unrelaxed_rank_001': '2', 'DI100047846_4lij-assembly2_unrelaxed_rank_002': '1', 'DI100047846_4lij-assembly2_unrelaxed_rank_003': '3', 'DI100047846_4lij-assembly2_unrelaxed_rank_004': '5', 'DI100047846_4lij-assembly2_unrelaxed_rank_005': '4', 'DI100047847_4lij-assembly2_unrelaxed_rank_001': '2', 'DI100047847_4lij-assembly2_unrelaxed_rank_002': '1', 'DI100047847_4lij-assembly2_unrelaxed_rank_003': '3', 'DI100047847_4lij-assembly2_unrelaxed_rank_004': '5', 'DI100047847_4lij-assembly2_unrelaxed_rank_005': '4', 'DI100062240_4ljp-assembly1_unrelaxed_rank_001': '4', 'DI100062240_4ljp-assembly1_unrelaxed_rank_002': '3', 'DI100062240_4ljp-assembly1_unrelaxed_rank_003': '1', 'DI100062240_4ljp-assembly1_unrelaxed_rank_004': '2', 'DI100062240_4ljp-assembly1_unrelaxed_rank_005': '5', 'DI100097909_5lk8-assembly1_unrelaxed_rank_001': '5', 'DI100097909_5lk8-assembly1_unrelaxed_rank_002': '3', 'DI100097909_5lk8-assembly1_

In [ ]:
# Calculate DockQ for all predicted models
# Took about 90 minutes to run on ethernet
from DockQ.DockQ import load_PDB, run_on_all_native_interfaces

model_filepath = "/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/output_files/"
model_filename1 = "_unrelaxed_rank_00"
model_filename2 = "_alphafold2_multimer_v3_model_"
model_fileext = "_seed_000.pdb"

scores_filepath = "/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/AlphaFold_benchmarking/output_files/"
scores_filename1 = "_scores_rank_00"
scores_filename2 = "_alphafold2_multimer_v3_model_"
scores_fileext = "_seed_000.json"

template_filepath = "/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/two-chain_dimerreps/"
template_fileext = ".cif"

for i,r in results_df.iterrows():

    lookup_id = r["report_lookup_id"]
    rank = r["rank"]
    model_num = rank_to_model[lookup_id + model_filename1 + rank]

    dockq_structure_af = load_PDB(model_filepath + lookup_id + model_filename1 + rank + model_filename2 + model_num + model_fileext)
    dockq_structure_solved = load_PDB(template_filepath + lookup_id + template_fileext)

    chainid1 = r["chain_id_0"]
    chainid2 = r["chain_id_1"]
    chains = sorted([chainid1, chainid2])
    chain_map = {chains[0]: "A", chains[1]:"B"}
    chain_key = chains[0] + chains[1]

    with open(scores_filepath + lookup_id + scores_filename1 + rank + scores_filename2 + model_num + scores_fileext, "r") as f:
        scores = json.load(f)
    curr_iptm = scores['iptm']

    try:
        result = run_on_all_native_interfaces(dockq_structure_af, dockq_structure_solved, chain_map=chain_map)[0]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "DockQ"] = result[chain_key]["DockQ"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "iRMSD"] = result[chain_key]["iRMSD"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "lRMSD"] = result[chain_key]["LRMSD"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "FNonNat"] = np.float64(result[chain_key]["fnonnat"])
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "nat_correct"] = result[chain_key]["nat_correct"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "nat_total"] = result[chain_key]["nat_total"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "nonnat_count"] = result[chain_key]["nonnat_count"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "model_total"] = result[chain_key]["model_total"]
        results_df.loc[((results_df["report_lookup_id"] == lookup_id) & (results_df["rank"] == rank)), "iptm"] = np.float64(curr_iptm)
        
    except:
        print("Unable to calculate DockQ for structure " + lookup_id)

/opt/anaconda3/envs/biopython/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unable to calculate DockQ for structure DI37660814_9dog-assembly1
Unable to calculate DockQ for structure DI43730862_8eze-assembly1
Unable to calculate DockQ for structure DI37660830_9dog-assembly1
Unable to calculate DockQ for structure DI43291046_8esr-assembly1
Unable to calculate DockQ for structure DI171349247_8qau-assembly1
Unable to calculate DockQ for structure DI208164626_7tkh-assembly1
Unable to calculate DockQ for structure DI56264438_8fnz-assembly1
Unable to calculate DockQ for structure DI172449219_7qkh-assembly1
Unable to calculate DockQ for structure DI207923253_8teq-assembly1
Unable to calculate DockQ for structure DI227981889_7vqq-assembly1
Unable to calculate DockQ for structure DI172449440_7qky-assembly1
Unable to calculate DockQ for structure DI40741427_8ec7-assembly1
Unable to calculate DockQ for structure DI205688668_8tau-assembly1
Unable to calculate DockQ for structure DI243809085_8wcp-assembly1
Unable to calculate DockQ for structure DI171336719_8q8z-assembly1
U

In [22]:
results_df["if_type"] = results_df["old_complex_id"].map(dict(zip(selected_ifs_concat["old_complex_id"], selected_ifs_concat["if_type"])))
results_df["protpep_status"] = results_df["old_complex_id"].map(dict(zip(selected_ifs_concat["old_complex_id"], selected_ifs_concat["protpep_status"])))
results_df["maxiflen"] = results_df["old_complex_id"].map(dict(zip(selected_ifs_concat["old_complex_id"], selected_ifs_concat["maxiflen"])))
results_df["miniflen"] = results_df["old_complex_id"].map(dict(zip(selected_ifs_concat["old_complex_id"], selected_ifs_concat["miniflen"])))
results_df

,old_complex_id,pdb_id,chain_id_0,chain_id_1,if_size_cat,AFstatus,dimer_len,model_len,report_lookup_id,rank,...,FNonNat,nat_correct,nat_total,nonnat_count,model_total,iptm,if_type,protpep_status,maxiflen,miniflen
0,208129618,8tid-assembly1,A,F,Large,Post-training,923,1287,DI208129618_8tid-assembly1,1,...,0.549451,41.0,90.0,50.0,91.0,0.79,Order-order,Protein-protein,39,32
1,198367644,8s5d-assembly1,A-43,A-117,Small,Post-training,330,362,DI198367644_8s5d-assembly1,1,...,1.000000,0.0,18.0,32.0,32.0,0.35,Order-order,Protein-protein,13,11
2,62511604,8gzu-assembly1,B9,N5,Large,Post-training,901,939,DI62511604_8gzu-assembly1,1,...,0.989362,1.0,20.0,93.0,94.0,0.32,Order-order,Protein-protein,39,28
3,45569345,8faz-assembly1,B,D,Small,Post-training,380,684,DI45569345_8faz-assembly1,1,...,1.000000,0.0,20.0,50.0,50.0,0.72,Order-order,Protein-protein,12,10
4,57535053,8g2z-assembly1,0T,MB,Medium,Post-training,695,741,DI57535053_8g2z-assembly1,1,...,1.000000,0.0,44.0,82.0,82.0,0.63,Order-order,Protein-protein,31,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10065,118893928,3mzw-assembly1,A,B,Medium,Pre-training,626,682,DI118893928_3mzw-assembly1,5,...,1.000000,0.0,57.0,45.0,45.0,0.16,Disorder-order,Protein-protein,27,26
10066,212462140,1upk-assembly1,A,B,Small,Pre-training,326,353,DI212462140_1upk-assembly1,5,...,0.050000,19.0,19.0,1.0,20.0,0.61,Disorder-order,Protein-peptide,16,4
10067,245294625,4wyp-assembly1,A,B,Small,Pre-training,253,250,DI245294625_4wyp-assembly1,5,...,1.000000,0.0,18.0,44.0,44.0,0.14,Disorder-order,Protein-protein,16,14
10068,115143524,2mgu-assembly1,A,M,Large,Pre-training,188,184,DI115143524_2mgu-assembly1,5,...,0.758929,27.0,90.0,85.0,112.0,0.22,Disorder-order,Protein-protein,58,31


In [23]:
results_df.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/afmm_benchmarking.tsv", sep="\t", index=None)

## Analyze results

In [11]:
results_df = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/afmm_benchmarking.tsv", sep="\t")
print("Number of rows without DockQ value:", results_df[results_df["DockQ"].isna()].shape[0])

Number of rows without DockQ value: 130


In [12]:
num_if_per_pdb = pdb_clusters.groupby("pdb_id")["old_complex_id"].size()
single_dimers = list(set(num_if_per_pdb[num_if_per_pdb == 1].index))
print(len(single_dimers))
pdb_clusters.loc[:, "single_dimer"] = 0
pdb_clusters.loc[pdb_clusters["pdb_id"].isin(single_dimers), "single_dimer"] = 1
print(pdb_clusters[pdb_clusters["single_dimer"] == 1].shape[0])

97200
97200


In [13]:
results_df["single_dimer"] = results_df["old_complex_id"].map(dict(zip(pdb_clusters.old_complex_id, pdb_clusters.single_dimer)))
results_df["pdb_resolution"] = results_df["old_complex_id"].map(dict(zip(pdb_clusters.old_complex_id, pdb_clusters.pdb_resolution)))
results_df["pdb_expt_method"] = results_df["old_complex_id"].map(dict(zip(pdb_clusters.old_complex_id, pdb_clusters.pdb_expt_method)))

In [14]:
results_df.columns

Index(['old_complex_id', 'pdb_id', 'chain_id_0', 'chain_id_1', 'if_size_cat',
       'AFstatus', 'dimer_len', 'model_len', 'report_lookup_id', 'rank',
       'model_to_nat_len_ratio', 'DockQ', 'iRMSD', 'lRMSD', 'FNonNat',
       'nat_correct', 'nat_total', 'nonnat_count', 'model_total', 'iptm',
       'if_type', 'protpep_status', 'maxiflen', 'miniflen', 'single_dimer',
       'pdb_resolution', 'pdb_expt_method'],
      dtype='object')

In [15]:
results_df.loc[results_df["DockQ"] >= 0, "DockQabove0.23"] = 0
results_df.loc[results_df["DockQ"] > 0.23, "DockQabove0.23"] = 1
top_rank_only = results_df[results_df["rank"] == 1]
results_df_nonan = results_df.dropna(subset=["DockQ"], axis=0)
top_rank_only_nonan = top_rank_only.dropna(subset=["DockQ"], axis=0)
top_rank_only_singdim = top_rank_only_nonan[top_rank_only_nonan["single_dimer"] == 1]

In [17]:
pre_train_scores = top_rank_only_singdim[top_rank_only_singdim["AFstatus"] == "Pre-training"]["DockQ"]
post_train_scores = top_rank_only_singdim[top_rank_only_singdim["AFstatus"] == "Post-training"]["DockQ"]

scipy.stats.ttest_ind(a=pre_train_scores, b=post_train_scores)

TtestResult(statistic=1.2553201120334845, pvalue=0.20997719154404718, df=477.0)

In [19]:
top_rank_only_nonan.groupby(["AFstatus","if_type"])["DockQabove0.23"].describe()

count      mean       std  min  25%  50%  \
AFstatus      if_type                                                       
Post-training Disorder-disorder  291.0  0.116838  0.321781  0.0  0.0  0.0   
              Disorder-order     365.0  0.232877  0.423245  0.0  0.0  0.0   
              Order-order        348.0  0.301724  0.459667  0.0  0.0  0.0   
Pre-training  Disorder-disorder  299.0  0.237458  0.426239  0.0  0.0  0.0   
              Disorder-order     348.0  0.341954  0.475047  0.0  0.0  0.0   
              Order-order        309.0  0.343042  0.475495  0.0  0.0  0.0   

                                 75%  max  
AFstatus      if_type                      
Post-training Disorder-disorder  0.0  1.0  
              Disorder-order     0.0  1.0  
              Order-order        1.0  1.0  
Pre-training  Disorder-disorder  0.0  1.0  
              Disorder-order     1.0  1.0  
              Order-order        1.0  1.0

In [22]:
top_rank_only_singdim.groupby(["AFstatus","if_type"])["DockQabove0.23"].describe()

count      mean       std  min  25%  50%  \
AFstatus      if_type                                                       
Post-training Disorder-disorder   22.0  0.227273  0.428932  0.0  0.0  0.0   
              Disorder-order      81.0  0.617284  0.489078  0.0  0.0  1.0   
              Order-order         91.0  0.483516  0.502497  0.0  0.0  0.0   
Pre-training  Disorder-disorder   95.0  0.526316  0.501956  0.0  0.0  1.0   
              Disorder-order      95.0  0.631579  0.484935  0.0  0.0  1.0   
              Order-order         95.0  0.547368  0.500392  0.0  0.0  1.0   

                                 75%  max  
AFstatus      if_type                      
Post-training Disorder-disorder  0.0  1.0  
              Disorder-order     1.0  1.0  
              Order-order        1.0  1.0  
Pre-training  Disorder-disorder  1.0  1.0  
              Disorder-order     1.0  1.0  
              Order-order        1.0  1.0

In [23]:
top_rank_only_singdim.groupby(["AFstatus","DockQabove0.23"])["old_complex_id"].count()

AFstatus       DockQabove0.23
Post-training  0.0                95
               1.0                99
Pre-training   0.0               123
               1.0               162
Name: old_complex_id, dtype: int64

In [24]:
scipy.stats.fisher_exact([[95,99],[123,162]])

SignificanceResult(statistic=1.2638580931263859, pvalue=0.22502601398982047)